In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf, numpy as np, random, os, glob
random.seed(0); np.random.seed(0); tf.random.set_seed(0)

!cp /content/drive/MyDrive/my_data.zip .
!unzip -q -o my_data.zip -d .
!wget -q http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip
!unzip -q -o mini_speech_commands.zip -d data
!wget -q http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
!mkdir -p sc02 && tar -xzf speech_commands_v0.02.tar.gz -C sc02
!ls my_data

Mounted at /content/drive
down  go  left	no  right  silence  stop  unknown  up  yes


In [ ]:
SR = 16000
FRAME_LEN, FRAME_STEP, FFT_LEN = 640, 320, 1024
N_MEL, N_MFCC, LOW_HZ, HIGH_HZ = 40, 10, 20, 4000

def get_mfcc(waveform):
    stft = tf.signal.stft(waveform, frame_length=FRAME_LEN, frame_step=FRAME_STEP, fft_length=FFT_LEN)
    spectrogram = tf.abs(stft)
    mel_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MEL, num_spectrogram_bins=FFT_LEN // 2 + 1,
        sample_rate=SR, lower_edge_hertz=LOW_HZ, upper_edge_hertz=HIGH_HZ)
    mel = tf.tensordot(spectrogram, mel_matrix, 1)
    log_mel = tf.math.log(mel + 1e-6)
    mfcc = tf.signal.mfccs_from_log_mel_spectrograms(log_mel)[..., :N_MFCC]
    return mfcc[..., tf.newaxis]

WORDS = ['down', 'go', 'left', 'no', 'right', 'stop', 'up', 'yes']
CLASSES = sorted(WORDS + ['silence', 'unknown'])
CID = {c: i for i, c in enumerate(CLASSES)}
print(CLASSES)

def load_wav(path):
    a = tf.audio.decode_wav(tf.io.read_file(path), desired_channels=1, desired_samples=SR)[0]
    return tf.squeeze(a, -1).numpy()

X, Y, SRC = [], [], []

# Google's 8 command words
for w in WORDS:
    for f in sorted(glob.glob(f'data/mini_speech_commands/{w}/*.wav')):
        X.append(load_wav(f)); Y.append(CID[w]); SRC.append('google')

# "unknown": 30 clips from each of the other words in the big dataset
skip = set(WORDS) | {'_background_noise_'}
others = [d for d in sorted(os.listdir('sc02')) if os.path.isdir(f'sc02/{d}') and d not in skip]
print(len(others), 'other words')
for d in others:
    files = sorted(glob.glob(f'sc02/{d}/*.wav'))
    random.shuffle(files)
    for f in files[:30]:
        X.append(load_wav(f)); Y.append(CID['unknown']); SRC.append('google')

# "silence": slices of background noise at random loudness
noises = [tf.squeeze(tf.audio.decode_wav(tf.io.read_file(f), desired_channels=1)[0], -1).numpy()
          for f in glob.glob('sc02/_background_noise_/*.wav')]
print(len(noises), 'noise files')
for _ in range(800):
    n = random.choice(noises)
    s = random.randint(0, len(n) - SR)
    X.append(n[s:s + SR] * random.uniform(0.05, 1.0)); Y.append(CID['silence']); SRC.append('google')

# your own recordings
for c in CLASSES:
    files = sorted(glob.glob(f'my_data/{c}/*.wav'))
    print('mine', c, len(files))
    for f in files:
        X.append(load_wav(f)); Y.append(CID[c]); SRC.append('mine')

['down', 'go', 'left', 'no', 'right', 'silence', 'stop', 'unknown', 'up', 'yes']
27 other words
6 noise files
mine down 41
mine go 41
mine left 41
mine no 41
mine right 41
mine silence 41
mine stop 41
mine unknown 41
mine up 41
mine yes 41


In [ ]:
X = np.stack(X).astype(np.float32); Y = np.array(Y); SRC = np.array(SRC)
print(X.shape, np.bincount(Y))

idx = np.arange(len(X)); np.random.shuffle(idx)
train_i, val_i, test_g, test_m = [], [], [], []
for i in idx:
    r = np.random.rand()
    if SRC[i] == 'google':
        (test_g if r < 0.1 else val_i if r < 0.2 else train_i).append(i)
    else:
        (test_m if r < 0.2 else val_i if r < 0.3 else train_i).append(i)

def augment(x):
    x = x * random.uniform(0.4, 2.5)                       # louder or quieter
    if random.random() < 0.6:                              # add some background noise
        n = random.choice(noises)
        s = random.randint(0, len(n) - SR)
        x = x + n[s:s + SR] * random.uniform(0.0, 0.15)
    return np.clip(x, -1.0, 1.0).astype(np.float32)

Xtr, Ytr = [], []
for i in train_i:
    Xtr.append(X[i]); Ytr.append(Y[i])
    for _ in range(4 if SRC[i] == 'mine' else 1):          # your clips get more copies
        Xtr.append(augment(X[i])); Ytr.append(Y[i])
Xtr = np.stack(Xtr); Ytr = np.array(Ytr)

def feats(A):
    return np.concatenate([get_mfcc(tf.constant(A[i:i + 256])).numpy() for i in range(0, len(A), 256)])

Ftr = feats(Xtr)
Fva, Yva = feats(X[val_i]), Y[val_i]
Fte_g, Yte_g = feats(X[test_g]), Y[test_g]
Fte_m, Yte_m = feats(X[test_m]), Y[test_m]
print(Ftr.shape, Fva.shape, Fte_g.shape, Fte_m.shape)

(10020, 16000) [1041 1041 1041 1041 1041  841 1041  851 1041 1041]
(16696, 49, 10, 1) (1017, 49, 10, 1) (986, 49, 10, 1) (86, 49, 10, 1)


In [ ]:
from tensorflow.keras import layers

def build_model(input_shape=(49, 10, 1), n_classes=len(CLASSES)):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, (10, 4), strides=(2, 2), padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    for _ in range(4):
        x = layers.DepthwiseConv2D((3, 3), padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(64, (1, 1), use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    return tf.keras.Model(inp, layers.Dense(n_classes)(x))

model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
early = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)
model.fit(Ftr, Ytr, validation_data=(Fva, Yva), epochs=40, batch_size=64, callbacks=[early])

print('Google test accuracy  :', model.evaluate(Fte_g, Yte_g, verbose=0)[1])
print('My-voice test accuracy:', model.evaluate(Fte_m, Yte_m, verbose=0)[1])
model.save('/content/drive/MyDrive/kws2.keras')

Epoch 1/40
261/261 ━━━━━━━━━━━━━━━━━━━━ 19s 32ms/step - accuracy: 0.5730 - loss: 1.3548 - val_accuracy: 0.0875 - val_loss: 2.3043
Epoch 2/40
261/261 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8327 - loss: 0.5747 - val_accuracy: 0.8673 - val_loss: 0.6960
Epoch 3/40
261/261 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8800 - loss: 0.4042 - val_accuracy: 0.8830 - val_loss: 0.3836
Epoch 4/40
261/261 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9017 - loss: 0.3244 - val_accuracy: 0.9007 - val_loss: 0.3284
Epoch 5/40
261/261 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.9161 - loss: 0.2793 - val_accuracy: 0.8879 - val_loss: 0.3601
Epoch 6/40
261/261 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9275 - loss: 0.2416 - val_accuracy: 0.8987 - val_loss: 0.3455
Epoch 7/40
261/261 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9323 - loss: 0.2180 - val_accuracy: 0.9066 - val_loss: 0.3194
Epoch 8/40
261/261 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9438 - loss: 0.1881 - val_accuracy

In [ ]:
pred = np.argmax(model.predict(Fte_m, verbose=0), axis=1)
for i, c in enumerate(CLASSES):
    m = Yte_m == i
    if m.sum():
        print(f'{c:8s} {np.mean(pred[m] == i):.2f}  ({m.sum()} clips)')

old = tf.keras.models.load_model('/content/drive/MyDrive/kws.keras')
map_old = np.array([CID[w] for w in WORDS])
mask = np.isin(Yte_m, map_old)
old_pred = map_old[np.argmax(old.predict(Fte_m[mask], verbose=0), axis=1)]
new_pred = np.argmax(model.predict(Fte_m[mask], verbose=0), axis=1)
print('OLD model on my word clips:', np.mean(old_pred == Yte_m[mask]))
print('NEW model on my word clips:', np.mean(new_pred == Yte_m[mask]))

down     1.00  (8 clips)
go       1.00  (11 clips)
left     1.00  (4 clips)
no       1.00  (7 clips)
right    1.00  (12 clips)
silence  0.83  (6 clips)
stop     1.00  (14 clips)
unknown  0.83  (6 clips)
up       1.00  (9 clips)
yes      0.89  (9 clips)
OLD model on my word clips: 0.918918918918919
NEW model on my word clips: 0.9864864864864865


In [ ]:
sel = np.random.choice(len(Ftr), 300, replace=False)
def rep():
    for j in sel:
        yield [Ftr[j:j + 1]]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = rep
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
int8_tflite = conv.convert()
open('kws2_int8.tflite', 'wb').write(int8_tflite)
print('int8 size KB:', round(len(int8_tflite) / 1024, 1))

def eval_int8(F, Yt):
    it = tf.lite.Interpreter(model_content=int8_tflite); it.allocate_tensors()
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    s, z = ind['quantization']
    ok = 0
    for f, y in zip(F, Yt):
        q = np.clip(np.round(f[np.newaxis] / s + z), -128, 127).astype(np.int8)
        it.set_tensor(ind['index'], q); it.invoke()
        ok += int(np.argmax(it.get_tensor(outd['index'])[0]) == y)
    return ok / len(Yt)

print('int8 Google test:', eval_int8(Fte_g, Yte_g))
print('int8 My-voice test:', eval_int8(Fte_m, Yte_m))

it = tf.lite.Interpreter(model_content=int8_tflite); it.allocate_tensors()
scale, zero = it.get_input_details()[0]['quantization']

def c_array(data, name):
    hx = [f'0x{b:02x}' for b in data]
    rows = [', '.join(hx[i:i + 16]) for i in range(0, len(hx), 16)]
    return (f'alignas(8) const unsigned char {name}[] = {{\n  ' + ',\n  '.join(rows)
            + f'\n}};\nconst unsigned int {name}_len = {len(data)};\n')

with open('model_data.h', 'w') as f:
    f.write('#pragma once\n')
    f.write(f'#define N_CLASSES {len(CLASSES)}\n')
    f.write(f'const float model_in_scale = {scale:.8e}f;\nconst int model_in_zero = {int(zero)};\n')
    f.write('const char* class_names[N_CLASSES] = {' + ','.join(f'"{c}"' for c in CLASSES) + '};\n')
    f.write(c_array(int8_tflite, 'kws_model'))

!cp kws2_int8.tflite model_data.h /content/drive/MyDrive/
from google.colab import files
files.download('model_data.h')

Saved artifact at '/tmp/tmpmfvsiwhi'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 10, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  134752228160336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228158800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228161872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228161104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228158992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228162256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228163216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228159760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228161296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228163024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134752228160144

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


int8 size KB: 44.6


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


int8 Google test: 0.8833671399594321
int8 My-voice test: 0.9651162790697675


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf, numpy as np, random, os, glob
random.seed(0); np.random.seed(0); tf.random.set_seed(0)

!cp /content/drive/MyDrive/my_data.zip .
!unzip -q -o my_data.zip -d .
!wget -q http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip
!unzip -q -o mini_speech_commands.zip -d data
!wget -q http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
!mkdir -p sc02 && tar -xzf speech_commands_v0.02.tar.gz -C sc02

SR = 16000
FRAME_LEN, FRAME_STEP, FFT_LEN = 640, 320, 1024
N_MEL, N_MFCC, LOW_HZ, HIGH_HZ = 40, 10, 20, 4000

def get_mfcc(waveform):
    stft = tf.signal.stft(waveform, frame_length=FRAME_LEN, frame_step=FRAME_STEP, fft_length=FFT_LEN)
    spectrogram = tf.abs(stft)
    mel_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MEL, num_spectrogram_bins=FFT_LEN // 2 + 1,
        sample_rate=SR, lower_edge_hertz=LOW_HZ, upper_edge_hertz=HIGH_HZ)
    mel = tf.tensordot(spectrogram, mel_matrix, 1)
    log_mel = tf.math.log(mel + 1e-6)
    mfcc = tf.signal.mfccs_from_log_mel_spectrograms(log_mel)[..., :N_MFCC]
    return mfcc[..., tf.newaxis]

WORDS = ['down', 'go', 'left', 'no', 'right', 'stop', 'up', 'yes']
CLASSES = sorted(WORDS + ['silence', 'unknown'])
CID = {c: i for i, c in enumerate(CLASSES)}
print(CLASSES)

def load_wav(path):
    a = tf.audio.decode_wav(tf.io.read_file(path), desired_channels=1, desired_samples=SR)[0]
    return tf.squeeze(a, -1).numpy()

X, Y, SRC = [], [], []
for w in WORDS:
    for f in sorted(glob.glob(f'data/mini_speech_commands/{w}/*.wav')):
        X.append(load_wav(f)); Y.append(CID[w]); SRC.append('google')

skip = set(WORDS) | {'_background_noise_'}
others = [d for d in sorted(os.listdir('sc02')) if os.path.isdir(f'sc02/{d}') and d not in skip]
for d in others:
    files = sorted(glob.glob(f'sc02/{d}/*.wav'))
    random.shuffle(files)
    for f in files[:30]:
        X.append(load_wav(f)); Y.append(CID['unknown']); SRC.append('google')

noises = [tf.squeeze(tf.audio.decode_wav(tf.io.read_file(f), desired_channels=1)[0], -1).numpy()
          for f in glob.glob('sc02/_background_noise_/*.wav')]
for _ in range(800):
    n = random.choice(noises)
    s = random.randint(0, len(n) - SR)
    X.append(n[s:s + SR] * random.uniform(0.05, 1.0)); Y.append(CID['silence']); SRC.append('google')

for c in CLASSES:
    files = sorted(glob.glob(f'my_data/{c}/*.wav'))
    for f in files:
        X.append(load_wav(f)); Y.append(CID[c]); SRC.append('mine')

X = np.stack(X).astype(np.float32); Y = np.array(Y); SRC = np.array(SRC)
print(X.shape, np.bincount(Y))

idx = np.arange(len(X)); np.random.shuffle(idx)
train_i, val_i, test_g, test_m = [], [], [], []
for i in idx:
    r = np.random.rand()
    if SRC[i] == 'google':
        (test_g if r < 0.1 else val_i if r < 0.2 else train_i).append(i)
    else:
        (test_m if r < 0.2 else val_i if r < 0.3 else train_i).append(i)

def feats(A):
    return np.concatenate([get_mfcc(tf.constant(A[i:i + 256])).numpy() for i in range(0, len(A), 256)])

Fte_g, Yte_g = feats(X[test_g]), Y[test_g]
Fte_m, Yte_m = feats(X[test_m]), Y[test_m]

model = tf.keras.models.load_model('/content/drive/MyDrive/kws2.keras')
int8_tflite = open('/content/drive/MyDrive/kws2_int8.tflite', 'rb').read()

print('Google test acc:', model.evaluate(Fte_g, Yte_g, verbose=0)[1])
print('My test acc:', model.evaluate(Fte_m, Yte_m, verbose=0)[1])

Mounted at /content/drive
['down', 'go', 'left', 'no', 'right', 'silence', 'stop', 'unknown', 'up', 'yes']
(10020, 16000) [1041 1041 1041 1041 1041  841 1041  851 1041 1041]
Google test acc: 0.8884381055831909
My test acc: 0.9651162624359131


In [ ]:
N = 50
idx = np.random.choice(len(Fte_g), min(N, len(Fte_g)), replace=False)
F = Fte_g[idx]; Y = Yte_g[idx]

interp = tf.lite.Interpreter(model_content=int8_tflite); interp.allocate_tensors()
in_d = interp.get_input_details()[0]; out_d = interp.get_output_details()[0]
scale, zero = in_d['quantization']

q_inputs, expected = [], []
for f in F:
    q = np.clip(np.round(f[np.newaxis] / scale + zero), -128, 127).astype(np.int8)
    interp.set_tensor(in_d['index'], q); interp.invoke()
    expected.append(int(np.argmax(interp.get_tensor(out_d['index'])[0])))
    q_inputs.append(q.flatten())
q_inputs = np.stack(q_inputs)
print(q_inputs.shape)
print('accuracy on these clips:', np.mean(np.array(expected) == Y))

def c_array(data, name):
    hx = [f'0x{b:02x}' for b in data]
    rows = [', '.join(hx[i:i+16]) for i in range(0, len(hx), 16)]
    return f'alignas(8) const unsigned char {name}[] = {{\n  ' + ',\n  '.join(rows) + f'\n}};\nconst unsigned int {name}_len = {len(data)};\n'

with open('model_data.h', 'w') as f:
    f.write('#pragma once\n')
    f.write(f'#define N_CLASSES {len(CLASSES)}\n')
    f.write(c_array(int8_tflite, 'kws_model'))

with open('test_data.h', 'w') as f:
    f.write('#pragma once\n#include <stdint.h>\n')
    f.write(f'#define N_TEST {len(q_inputs)}\n#define INPUT_SIZE {q_inputs.shape[1]}\n')
    f.write('const int8_t test_inputs[N_TEST][INPUT_SIZE] = {\n')
    for row in q_inputs:
        f.write('  {' + ','.join(str(int(v)) for v in row) + '},\n')
    f.write('};\n')
    f.write('const uint8_t test_labels[N_TEST] = {' + ','.join(map(str, Y)) + '};\n')
    f.write('const uint8_t expected_pred[N_TEST] = {' + ','.join(map(str, expected)) + '};\n')

from google.colab import files
files.download('model_data.h')
files.download('test_data.h')

(50, 490)
accuracy on these clips: 0.86


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>